# QLoRA fine-tuning of DeepSeek-R1-Distill-Qwen-1.5B for medical reasoning

PEFT-QLoRA example fine-tuning `FreedomIntelligence/medical-o1-reasoning-SFT` on the lightweight
`deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` model so it fits on a free T4 GPU (16GB).

> **v2 changes**: fixed tokenizer format (model uses its real full-width special tokens +
> `<think>` tags, not `<|user|>`/`<|end|>`), T4-correct precision (`fp16`, not `bf16`),
> current `trl` API (`SFTConfig`, `max_length`, `dataset_text_field="text"`), removed
> `trust_remote_code`, correct adapter save path, added inference demo. See last cell for changelog.

In [ ]:
# 1. Install and import
!pip install -q -U "accelerate>=0.33" "peft>=0.13" "transformers>=4.46" "datasets>=2.21" "bitsandbytes>=0.43" "trl>=0.19"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training
from datasets import load_dataset

In [ ]:
# 2. Model + QLoRA config
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# NF4 (Normal Float 4) quantization
# bnb_4bit_compute_dtype=torch.float16: T4 is Turing (sm_75) - it has NO native bf16
# tensor cores, so bf16 is emulated and much slower. fp16 is the correct T4 default.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # extra memory savings, no quality loss
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",       # fast attention, works on T4
)

# trust_remote_code removed: this model needs no custom code, only lowers trust surface.
tokenizer = AutoTokenizer.from_pretrained(model_name)
# This tokenizer already defines pad == eos; set explicitly to be safe.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("special tokens:", tokenizer.special_tokens_map)

In [ ]:
# 3. Load dataset (en split, first 5000 rows)
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", name="en", split="train[:5000]")
print("columns:", dataset.column_names)
print("rows:", len(dataset))
print("question:", dataset[0]["Question"][:120])

In [ ]:
# 4. Formatting using the model's REAL special tokens
# DeepSeek-R1-Distill-Qwen's tokenizer uses the full-width characters
# U+FF5C (你) and U+2581 (你) around role names, plus the <think>/</think> tags.
# The previous hardcoded `<|user|>` / `<|end|>` (ASCII) tokens are NOT in this
# model's vocabulary and get split into unrelated subword tokens - silently
# corrupting training data. Verified against tokenizer.json: all tokens below
# are single added tokens.
USER = "<\uff5cUser\uff5c>"
ASSISTANT = "<\uff5cAssistant\uff5c>"
EOS = "<\uff5cend\u2581of\u2581sentence\uff5c>"

def format_chat_template(example):
    return (
        f"{USER}{example['Question']}{EOS}\n"
        f"{ASSISTANT}<think>\n{example['Complex_CoT']}\n</think>\n{example['Response']}{EOS}"
    )

# Map to a plain "text" column and use dataset_text_field="text" (the default).
# This is the documented path in every trl release, avoiding the
# formatting_func/dataset_text_field interplay that changed between versions.
dataset = dataset.map(format_chat_template)

# sanity check: special tokens must tokenize as single tokens
sample = dataset[0]["text"]
print(tokenizer.tokenize(sample)[:8])
print("total tokens in first example:", len(tokenizer.tokenize(sample)))

In [ ]:
# 5. LoRA adapter + prep for k-bit training
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# gradient_checkpointing_kwargs: torch>=2.9 raises if use_reentrant is not explicit
model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

In [ ]:
# 6. Train with SFTTrainer
training_args = SFTConfig(
    output_dir="./medical-qlora-results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_steps=50,
    save_total_limit=2,          # don't fill the disk with checkpoints
    logging_steps=25,
    learning_rate=2e-4,
    warmup_steps=2,
    fp16=True,                   # T4: fp16, NOT bf16
    bf16=False,
    optim="paged_adamw_8bit",    # less VRAM for the optimizer on T4
    report_to="none",
    seed=42,
    max_length=2048,             # model_max_length is 16k; 1024 default truncates long CoTs
    packing=False,
    # dataset_text_field defaults to "text" - the column we created in cell 4
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
# 7. Save the adapter (NOT the base model!)
# `model` here still points at the unwrapped 4-bit base model; SFTTrainer wraps it
# into a PeftModel internally. Saving the outer `model` would save the base weights,
# not the adapter - previously the "LoRA adapter saved" message was wrong.
trainer.model.save_pretrained("./medical-qlora-adapter")
tokenizer.save_pretrained("./medical-qlora-adapter")
print("fine-tuning complete, LoRA adapter saved to ./medical-qlora-adapter")

In [ ]:
# 8. Inference: load adapter and test on a held-out question
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)
tok = AutoTokenizer.from_pretrained("./medical-qlora-adapter")
ft_model = PeftModel.from_pretrained(base, "./medical-qlora-adapter")

q = dataset[4]["Question"]
messages = [{"role": "user", "content": q}]
# add_generation_prompt=True appends the native "<\uff5cAssistant\uff5c> thinking\n"
# prefix, matching the training format from cell 4.
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(ft_model.device)

# DeepSeek recommends temperature 0.5-0.7 and enforcing thinking at generation start
out = ft_model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.6, top_p=0.95)
print(tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## What was fixed in v2

| # | Issue | Fix |
|---|-------|-----|
| 1 | **Wrong chat format**: used `<\|user\|>`/`<\|end\|>` (ASCII) tokens not in this model's vocab | Use native full-width tokens + `<think>` tags (cell 4) |
| 2 | **bf16 on T4**: T4 (sm_75) has no bf16 tensor cores; emulated bf16 is 2-4x slower / errors | `fp16=True`, `bnb_4bit_compute_dtype=torch.float16` (cells 2, 6) |
| 3 | **`trust_remote_code=True`** unnecessary and increases attack surface | Removed (cell 2) |
| 4 | **Broken with current `trl`**: `TrainingArguments` + missing `max_length`/dataset column crashes on trl>=0.19 | Use `SFTConfig` + pre-mapped `text` column (cell 6) |
| 5 | **Saved wrong artifact**: `model.save_pretrained()` saved the base 4-bit model, not the adapter | `trainer.model.save_pretrained()` (cell 7) |
| 6 | `use_reentrant` warning -> hard error on torch>=2.9 | Passed `gradient_checkpointing_kwargs={"use_reentrant": False}` (cell 5) |
| 7 | No `double_quant`, no `save_total_limit`, no seed, no inference check | Added (cells 2, 6, 8) |
| 8 | README size claim wrong: 1.5B-param FP16 is ~3.5GB, not ~6GB | README corrected |